# 01 – Data Audit (EVISION 360)

**අරමුණ:** Raw datasets තුන load කරලා, clean කරන්න කලින් ඒවායේ quality, structure සහ coverage පරීක්ෂා කිරීම.
මේ notebook එකෙන් raw files වෙනස් කරන්නේ නැහැ (read-only audit).

1. Setup සහ load
2. Ember audit
3. IEA audit
4. World Bank audit
5. Cross-source coverage (රටවල් ගැලපෙනවාද?)
6. Ember vs IEA reconciliation
7. Findings summary

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

RAW = Path("../data/raw")
EMBER_PATH = RAW / "ember" / "monthly_ev_sales.csv"
IEA_PATH   = RAW / "iea" / "EV data by country 2026.xlsx"
WB_PATH    = RAW / "worldbank" / "wdi_indicators_2010_2025.xlsx"
OUT = Path("../reports/insights")
OUT.mkdir(parents=True, exist_ok=True)

for p in (EMBER_PATH, IEA_PATH, WB_PATH):
    print(p.name, "->", "OK" if p.exists() else "MISSING")

monthly_ev_sales.csv -> OK
EV data by country 2026.xlsx -> OK
wdi_indicators_2010_2025.xlsx -> OK


In [2]:
ember  = pd.read_csv(EMBER_PATH)
iea    = pd.read_excel(IEA_PATH, sheet_name="GEVO_EV_2026")
wb_raw = pd.read_excel(WB_PATH, sheet_name="Data")

def profile(df, name):
    print(f"=== {name} ===")
    print("Shape:", df.shape)
    miss = df.isna().mean().sort_values(ascending=False).round(3)
    print("\nMissing share (top 8):\n", miss.head(8))
    print("\nFull-row duplicates:", df.duplicated().sum())

profile(ember, "Ember")
profile(iea, "IEA (GEVO_EV_2026)")
profile(wb_raw, "World Bank (raw)")

=== Ember ===
Shape: (46688, 24)

Missing share (top 8):
 Registrations YoY change (%)                   0.189
Registrations YoY change                       0.120
Rolling 12-month share of registrations (%)    0.103
Rolling 12-month registrations                 0.103
ISO 3 code                                     0.022
Continent                                      0.022
Ember region                                   0.022
EU member                                      0.022
dtype: float64

Full-row duplicates: 0
=== IEA (GEVO_EV_2026) ===
Shape: (49375, 9)

Missing share (top 8):
 unit              0.457
region_country    0.022
category          0.000
mode              0.000
parameter         0.000
powertrain        0.000
year              0.000
value             0.000
dtype: float64

Full-row duplicates: 15
=== World Bank (raw) ===
Shape: (1335, 20)

Missing share (top 8):
 Country Code     0.004
Series Name      0.004
Series Code      0.004
2010 [YR2010]    0.004
2018 [YR2018]    

## 2. Ember audit

In [3]:
# 2.1 Area type / frequency
print(ember["Area type"].value_counts(), "\n")
print(ember["Data frequency"].value_counts(), "\n")

# Region rows (EU, World) - ISO3 නැහැ, country analysis වලින් අයින් කරන්න ඕනේ
regions = ember[ember["Area type"] == "Region"]
print("Region areas:", regions["Area"].unique())
print("Countries/economies:", ember.loc[ember["Area type"] != "Region", "Area"].nunique())

# Quarterly rows කුමන areas වලටද?
print("\nQuarterly areas:", ember.loc[ember["Data frequency"] == "Quarterly", "Area"].unique())

Area type
Country or economy    45683
Region                 1005
Name: count, dtype: int64 

Data frequency
Monthly      46300
Quarterly      388
Name: count, dtype: int64 

Region areas: <ArrowStringArray>
['EU', 'World']
Length: 2, dtype: str
Countries/economies: 59

Quarterly areas: <ArrowStringArray>
['Canada', 'South Africa']
Length: 2, dtype: str


In [4]:
# 2.2 Drivetrain categories and aggregate flag
print(ember[["Drivetrain", "Is aggregate drivetrain"]].drop_duplicates().sort_values("Is aggregate drivetrain"))

# Double counting demo: all rows vs non-aggregate rows (countries, monthly, 2023)
m = ember[(ember["Area type"] != "Region") & (ember["Data frequency"] == "Monthly")
          & (ember["Date"].str[:4] == "2023")]
total_all = m["Registrations"].sum()
total_components = m.loc[~m["Is aggregate drivetrain"], "Registrations"].sum()
print(f"\nSum of ALL rows:           {total_all:,.0f}")
print(f"Sum of component rows only: {total_components:,.0f}")
print(f"Naive sum overstates by {total_all / total_components - 1:.0%}")

                       Drivetrain  Is aggregate drivetrain
0                Battery electric                    False
1                  Plug-in hybrid                    False
2                          Hybrid                    False
3                          Petrol                    False
4                          Diesel                    False
5                           Other                    False
3509         Ethanol/petrol blend                    False
5738     Combustion (unspecified)                    False
6     Electric vehicle (BEV+PHEV)                     True
7                 All drivetrains                     True

Sum of ALL rows:           152,676,128
Sum of component rows only: 69,749,044
Naive sum overstates by 119%


In [5]:
# 2.3 Key uniqueness and date coverage
key = ["Area", "Date", "Drivetrain", "Data frequency"]
print("Duplicate keys:", ember.duplicated(key).sum())
print("Date range:", ember["Date"].min(), "->", ember["Date"].max())

c = ember[ember["Area type"] != "Region"]
span = c.groupby("Area")["Date"].agg(first="min", last="max")
print("\nFirst-record year by country:")
print(span["first"].str[:4].value_counts().sort_index())
print("\nLast-record month by country:")
print(span["last"].value_counts().sort_index())

Duplicate keys: 0
Date range: 2005-01-01 -> 2026-10-01

First-record year by country:
first
2005     2
2006     1
2007     1
2010     2
2011     1
2012     3
2013     2
2015     5
2016     3
2017     3
2018     7
2019     5
2020     4
2021     1
2023    17
2024     2
Name: count, dtype: int64

Last-record month by country:
last
2026-06-01     3
2026-07-01     1
2026-08-01     4
2026-09-01    45
2026-10-01     6
Name: count, dtype: int64


**සටහන:** Ember වල 2026 data Oct 2026 දක්වා පමණයි (partial year). Annual analysis වලදී 2026 අයින් කරන්න හෝ YTD ලෙස පමණක් භාවිතා කරන්න.

## 3. IEA audit

In [6]:
# 3.1 Aggregates vs countries
AGG_GROUPS = {"_World", "European Union", "Other_aggregate", "Projection_region"}
names = iea.dropna(subset=["region_country"]).drop_duplicates("region_country")[["region_country", "Aggregate group"]]
iea_aggs = sorted(names.loc[names["Aggregate group"].isin(AGG_GROUPS), "region_country"])
iea_countries = sorted(names.loc[~names["Aggregate group"].isin(AGG_GROUPS), "region_country"])
print("Aggregates (%d):" % len(iea_aggs), iea_aggs)
print("Countries  (%d)" % len(iea_countries))
print("Sri Lanka in IEA?", "Sri Lanka" in iea_countries)

Aggregates (10): ['Advanced Economies', 'Africa', 'Asia Pacific', 'Developing Economies excl. China', 'Europe', 'European Union', 'Latin America', 'Middle East and Caspian', 'Southeast Asia', 'World']
Countries  (62)
Sri Lanka in IEA? False


In [7]:
# 3.2 Rows with no region/country (price data?)
nr = iea[iea["region_country"].isna()]
print("Rows without region_country:", len(nr))
print(nr["parameter"].value_counts())

# 3.3 Duplicates
dupe_key = ["region_country", "category", "parameter", "mode", "powertrain", "year"]
print("\nFull duplicate rows:", iea.duplicated().sum())
print("Duplicate keys:", iea.duplicated(dupe_key).sum())
print(iea[iea.duplicated(keep=False)].sort_values(dupe_key).head(6))

Rows without region_country: 1078
parameter
price_p0_2025USD      154
price_p100_2025USD    154
price_p25_2025USD     154
price_p50_2025USD     154
price_p75_2025USD     154
price_p95_2025USD     154
price_mean_2025USD    154
Name: count, dtype: int64

Full duplicate rows: 15
Duplicate keys: 15
             region_country    category                parameter mode powertrain  year   unit  value Aggregate group
19623  United Arab Emirates  Historical  Average capacity per EV   EV         EV  2021  kW/EV  0.310           Other
19805  United Arab Emirates  Historical  Average capacity per EV   EV         EV  2021  kW/EV  0.310           Other
19663  United Arab Emirates  Historical  Average capacity per EV   EV         EV  2022  kW/EV  0.120           Other
19808  United Arab Emirates  Historical  Average capacity per EV   EV         EV  2022  kW/EV  0.120           Other
19703  United Arab Emirates  Historical  Average capacity per EV   EV         EV  2023  kW/EV  0.069           Other
19

In [8]:
# 3.4 Categories, parameters, units, projections
print(iea.groupby("category")["year"].agg(["min", "max", "count"]), "\n")
print(iea.groupby(["parameter", "unit"], dropna=False).size(), "\n")

proj = iea[iea["category"].str.startswith("Projection")]
print("Projection rows:", len(proj), "| regions covered:", sorted(proj["region_country"].dropna().unique()))

                   min   max  count
category                           
Historical        2010  2025  48114
Projection-CPS    2035  2035    621
Projection-STEPS  2035  2035    640 

parameter                         unit                              
Average capacity per EV           kW/EV                                  601
Average capacity per EVSE         kW/EVSE                                602
Battery deployment                GWh                                    546
EV charging points                charging points                       1457
EV sales                          Vehicles                              7647
EV sales share                    percent                               2833
EV stock                          Vehicles                              7891
EV stock share                    percent                               2777
Electric LDVs per charging point  vehicles                               601
Electricity demand                GWh                   

In [9]:
# 3.5 Charging infrastructure coverage
ch = iea[(iea["parameter"] == "EV charging points") & (iea["category"] == "Historical")]
print("Countries/regions with charging data:", ch["region_country"].nunique())
print(ch.groupby("year")["region_country"].nunique().tail(8))
print("\nCharging powertrain types:", ch["powertrain"].unique())

Countries/regions with charging data: 53
year
2018    39
2019    40
2020    41
2021    50
2022    51
2023    52
2024    53
2025    53
Name: region_country, dtype: int64

Charging powertrain types: <ArrowStringArray>
['Publicly available fast', 'Publicly available slow', 'Publicly available ultra-fast']
Length: 3, dtype: str


## 4. World Bank audit

In [10]:
# 4.1 Footer / empty rows and basic cleaning (audit only)
wb = wb_raw[wb_raw["Country Code"].notna()].copy()
print("Raw rows:", len(wb_raw), "| after dropping footer rows:", len(wb))

year_cols = [c for c in wb.columns if "[YR" in str(c)]
wb[year_cols] = wb[year_cols].replace("..", np.nan).apply(pd.to_numeric, errors="coerce")
print("Series:", wb["Series Code"].unique())
print("Economies:", wb["Country Code"].nunique())

Raw rows: 1335 | after dropping footer rows: 1330
Series: <ArrowStringArray>
['NY.GDP.PCAP.CD', 'NY.GDP.MKTP.KD.ZG', 'SP.POP.TOTL', 'SP.URB.TOTL.IN.ZS', 'FP.CPI.TOTL.ZG']
Length: 5, dtype: str
Economies: 266


In [11]:
# 4.2 Coverage by indicator and year (% of economies with a value)
cov = wb.groupby("Series Code")[year_cols].agg(lambda s: s.notna().mean() * 100).round(1)
cov.columns = [c[:4] for c in cov.columns]
print(cov[["2010", "2015", "2020", "2023", "2024", "2025"]])

                   2010  2015  2020  2023  2024  2025
Series Code                                          
FP.CPI.TOTL.ZG     88.7  89.1  85.3  84.6  84.2  81.2
NY.GDP.MKTP.KD.ZG  97.0  97.4  97.0  94.7  93.2  88.0
NY.GDP.PCAP.CD     98.1  97.7  97.0  94.7  93.2  88.0
SP.POP.TOTL        99.6  99.6  99.6  99.6  99.6  99.6
SP.URB.TOTL.IN.ZS  99.6  99.6  99.6  99.6  99.6  99.6


**Aggregates:** World Bank file එකේ `World`, `Upper middle income`, `OECD members` වැනි aggregates තියෙනවා. ISO3 list එකක් එක්ක country rows විතරක් filter කරන්නේ Step 3 (country mapping) එකේදී.

## 5. Cross-source coverage

In [12]:
ember_countries = (ember[ember["ISO 3 code"].notna()]
                   .drop_duplicates("ISO 3 code")[["Area", "ISO 3 code"]])
wb_codes = set(wb["Country Code"])

missing_in_wb = ember_countries[~ember_countries["ISO 3 code"].isin(wb_codes)]
print("Ember countries missing in World Bank:\n", missing_in_wb)

# Sri Lanka check
print("\nSri Lanka -> Ember:", "LKA" in set(ember_countries["ISO 3 code"]),
      "| IEA:", "Sri Lanka" in iea_countries,
      "| World Bank:", "LKA" in wb_codes)

Ember countries missing in World Bank:
                  Area ISO 3 code
39792  Taiwan (China)        TWN

Sri Lanka -> Ember: False | IEA: False | World Bank: True


In [13]:
# Name mismatches Ember vs IEA (IEA has no ISO3 column)
ember_names = set(ember_countries["Area"])
iea_names = set(iea_countries)
only_ember = sorted(ember_names - iea_names)
only_iea = sorted(iea_names - ember_names)
print("In Ember only:", only_ember)
print("In IEA only :", only_iea)
print("\n-> country_mapping.csv (name -> ISO3) Step 3 වලදී හදන්න ඕනේ.")

In Ember only: ['Albania', 'Czechia', 'Ecuador', 'Hong Kong (SAR of China)', 'Malta', 'Peru', 'South Korea', 'Taiwan (China)', 'Türkiye', 'Ukraine', 'United States']
In IEA only : ['Argentina', 'Cambodia', 'Costa Rica', 'Czech Republic', 'Jordan', 'Korea', 'Lao PDR', 'Philippines', 'Russia', 'Seychelles', 'Turkiye', 'USA', 'United Arab Emirates', 'Uzbekistan']

-> country_mapping.csv (name -> ISO3) Step 3 වලදී හදන්න ඕනේ.


## 6. Ember vs IEA reconciliation (EV car sales, BEV+PHEV)

In [14]:
# Known name aliases so we can compare (full mapping comes in Step 3)
ALIAS = {"USA": "United States", "Korea": "South Korea", "Turkiye": "Türkiye",
         "Czech Republic": "Czechia"}

ember_ev = (ember[(ember["Drivetrain"] == "Electric vehicle (BEV+PHEV)")
                  & (ember["Data frequency"] == "Monthly")
                  & (ember["Area type"] != "Region")]
            .assign(year=lambda d: d["Date"].str[:4].astype(int))
            .groupby(["Area", "year"])["Registrations"].sum()
            .rename("ember").reset_index())

iea_ev = (iea[(iea["parameter"] == "EV sales") & (iea["category"] == "Historical")
              & (iea["mode"] == "Cars") & (iea["powertrain"].isin(["BEV", "PHEV"]))
              & iea["region_country"].isin(iea_countries)]
          .drop_duplicates(dupe_key)
          .assign(Area=lambda d: d["region_country"].replace(ALIAS))
          .groupby(["Area", "year"])["value"].sum()
          .rename("iea").reset_index())

rec = ember_ev.merge(iea_ev, on=["Area", "year"], how="inner")
rec["diff_pct"] = (rec["ember"] / rec["iea"] - 1) * 100
print("Comparable country-years:", len(rec))

# Only meaningful volumes (>10k) to avoid noise from tiny markets
big = rec[(rec["iea"] > 10_000) & (rec["year"].between(2018, 2024))]
print("\nMedian |difference| (%):", big["diff_pct"].abs().median().round(1))
print("Share of country-years within +/-10%:", (big["diff_pct"].abs() <= 10).mean().round(2))
print("\nLargest gaps:")
print(big.reindex(big["diff_pct"].abs().sort_values(ascending=False).index).head(10).round(1))

rec.round(2).to_csv(OUT / "ember_vs_iea_reconciliation.csv", index=False)

Comparable country-years: 374

Median |difference| (%): 1.1
Share of country-years within +/-10%: 0.85

Largest gaps:
               Area  year    ember      iea  diff_pct
332        Thailand  2019    691.0  10300.0     -93.3
36           Brazil  2021   2860.0  13900.0     -79.4
1         Australia  2021   8521.0  20400.0     -58.2
37           Brazil  2022   8440.0  18500.0     -54.4
225           Nepal  2024  17859.0  12054.0      48.2
281     South Korea  2018  31160.0  59400.0     -47.5
345  United Kingdom  2018  35165.0  62000.0     -43.3
169          Israel  2022  58706.0  43000.0      36.5
283     South Korea  2020  52947.0  39700.0      33.4
65         Colombia  2024   9178.0  13100.0     -29.9


## 7. Findings summary

In [15]:
summary = {
    "ember_rows": len(ember),
    "ember_countries": ember.loc[ember["Area type"] != "Region", "Area"].nunique(),
    "ember_region_rows": int((ember["Area type"] == "Region").sum()),
    "ember_quarterly_rows": int((ember["Data frequency"] == "Quarterly").sum()),
    "iea_rows": len(iea),
    "iea_aggregates": len(iea_aggs),
    "iea_countries": len(iea_countries),
    "iea_full_duplicates": int(iea.duplicated().sum()),
    "iea_rows_without_country": int(iea["region_country"].isna().sum()),
    "wb_economies": wb["Country Code"].nunique(),
    "ember_countries_missing_in_wb": ", ".join(missing_in_wb["Area"]),
    "sri_lanka_in_ember_or_iea": ("LKA" in set(ember_countries["ISO 3 code"])) or ("Sri Lanka" in iea_countries),
    "sri_lanka_in_wb": "LKA" in wb_codes,
}
pd.Series(summary, name="value").to_frame()

,value
ember_rows,46688
ember_countries,59
ember_region_rows,1005
ember_quarterly_rows,388
iea_rows,49375
iea_aggregates,10
iea_countries,62
iea_full_duplicates,15
iea_rows_without_country,1078
wb_economies,266
